In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from collections.abc import Callable
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker


In [7]:
P = 1000
MAX_N = 50
MAX_DEG = 25
STEP = 1
P = 1000
USE_LOG = True

In [3]:
def prepare_function(k, m) -> Callable:
    return lambda x: np.pow(np.e, -k * np.sin(m*x)) + k * np.sin(m*x) - 1

function = prepare_function(k=1, m=2)
interval = (np.float64(0), np.float64(3*np.pi))
y_lim = (-1.5, 2)

xs = np.linspace(interval[0], interval[1], 1000)
ys = [function(x) for x in xs]
plt.ylim(y_lim)
plt.xlabel('x')
plt.ylabel('y')
plt.title("Wizualizacja wykresu funkcji bazowej")
plt.plot(xs, ys, label="Funkcja bazowa")
plt.legend()
plt.savefig("base_function")
plt.close()

Funkcje pomocnicze

In [4]:
def generate_uniform_nodes(n: int, interval: tuple[np.float64, np.float64], f: Callable[[np.float64], np.float64]) -> np.ndarray:
    xs = np.linspace(interval[0], interval[1], n)
    return np.array([(xs[i], f(xs[i])) for i in range(n)], dtype=np.float64)

def generate_chebyshev_nodes(n: int, interval: tuple[np.float64, np.float64], f: Callable[[np.float64], np.float64]) -> np.ndarray:
    a, b = interval[0], interval[1]
    n -= 2
    roots = [-1] + [np.cos((2*k - 1)/(2 * n) * np.pi) for k in range(1, n+1)] + [1]
    xs = [(a+b)/2 + (b-a)/2 * root for root in roots]
    return np.array([(xs[i], f(xs[i])) for i in range(n+2)], dtype=np.float64)

def max_error(function, approximation, interval):
  xs = np.linspace(interval[0], interval[1], P)
  return np.round(max([np.abs(function(x) - approximation(x)) for x in xs]), 6)

def root_mean_squared_error(function, approximation, interval):
  xs = np.linspace(interval[0], interval[1], P)
  return np.round(np.sqrt(sum([(function(x) - approximation(x))**2 for x in xs]) / P), 6)

Aproksymacja trygonometryczna

In [5]:
def trigonometric_approximation(nodes: np.ndarray, degree: int) -> Callable[[np.float64], np.float64]:
    m = degree
    x_coords = nodes[:, 0]
    y_coords = nodes[:, 1]
    n = len(x_coords)

    if len(nodes) <= 2*m:
        raise Exception("Number of nodes must be greater than 2*degree")

    v_coords = ((x_coords + 5) / 10) * 2 * np.pi
    ks = np.arange(0, m + 1)
    kv = np.outer(ks, v_coords)

    a_k = (2 / n) * (np.cos(kv) @ y_coords)
    b_k = (2 / n) * (np.sin(kv) @ y_coords)
    a_k[0] /= 2

    def approximation(x: np.float64) -> np.float64:
        x1 = ((x + 5) / 10) * 2 * np.pi
        kx = ks * x1
        return np.float64(
            a_k[0] + np.sum(a_k[1:] * np.cos(kx[1:]) + b_k[1:] * np.sin(kx[1:]))
        )

    return approximation

Benchmark aproksymacji

In [8]:
metrics = ["max", "rmse"]

for metric in metrics:
    metric_function   = max_error if metric == "max" else root_mean_squared_error
    metric_name = "Max Error" if metric == "max" else "RMSE"

    def compute_error_matrix(node_generator):
        ns = range(2, MAX_N + 1, STEP)
        degs = range(1, MAX_DEG + 1, STEP)
        matrix = np.full((len(degs), len(ns)), np.nan)
        for ci, n in enumerate(ns):
            nodes = node_generator(n, interval, function)
            for ri, deg in enumerate(degs):
                if 2*deg >= n:
                    continue
                try:
                    approx = trigonometric_approximation(nodes, deg)
                    err = metric_function(function, approx, interval)
                    if np.isfinite(err):
                        matrix[ri, ci] = err
                except Exception:
                    pass
        return matrix, list(ns), list(degs)

    def plot_heatmap(ax, matrix, ns, degs, title, vmin, vmax):
        norm   = mcolors.LogNorm(vmin=max(vmin,1e-10), vmax=max(vmax,1e-9)) if USE_LOG else mcolors.Normalize(vmin=vmin, vmax=vmax)
        masked = np.ma.masked_invalid(matrix)

        im = ax.imshow(masked, origin="lower", aspect="auto", norm=norm, cmap="plasma")

        tick_step = max(1, len(ns) // 20) 
        x_idx = range(0, len(ns),   tick_step)
        y_idx = range(0, len(degs), tick_step)

        ax.set_xticks(list(x_idx))
        ax.set_xticklabels([ns[i]   for i in x_idx], fontsize=7)
        ax.set_yticks(list(y_idx))
        ax.set_yticklabels([degs[i] for i in y_idx], fontsize=7)
        ax.set_title(title, fontsize=11)
        ax.set_xlabel("Liczba węzłów", fontsize=9)
        ax.set_ylabel("Stopień aproksymacji", fontsize=9)
        ax.tick_params(labelsize=8)
        return im

    mat_u, ns, degs = compute_error_matrix(generate_uniform_nodes)
    mat_c, _,  _ = compute_error_matrix(generate_chebyshev_nodes)

    all_vals = np.concatenate([mat_u[np.isfinite(mat_u)], mat_c[np.isfinite(mat_c)]])
    vmin, vmax = np.percentile(all_vals, 1), np.percentile(all_vals, 99)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle(f"{metric_name}{' w skali logarytmicznej' if USE_LOG else ''}", fontsize=11)

    im1 = plot_heatmap(axes[0], mat_u, ns, degs, "równoodległe węzły", vmin, vmax)
    im2 = plot_heatmap(axes[1], mat_c, ns, degs, "węzły Czebyszewa", vmin, vmax)

    cbar = fig.colorbar(im2, ax=axes, fraction=0.03, pad=0.02)
    cbar.set_label(metric_name, fontsize=9)
    cbar.ax.tick_params(labelsize=8)
    cbar.ax.yaxis.set_major_locator(ticker.LogLocator(base=10, numticks=10))
    cbar.ax.yaxis.set_major_formatter(ticker.LogFormatterSciNotation())
    cbar.ax.yaxis.set_minor_locator(ticker.LogLocator(base=10, subs=np.arange(2, 10), numticks=50))
    cbar.ax.yaxis.set_minor_formatter(ticker.LogFormatterSciNotation(minor_thresholds=(2, 0.5)))
    cbar.ax.tick_params(which="both", labelsize=7)

    plt.savefig(metric)
    plt.close(fig)